In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene
import statsmodels.stats.multitest as smm

import sys
sys.path.append('../../')
sys.path.append('../../../')

from utilis import *
from settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

# Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
print(df.shape) 
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)
print(df.shape) 

In [ ]:
(df['p-tau217'] > HIGH_TAU_217_THR).value_counts()

# Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
# removing samples with moca below the minimum threshold of 19
df = df[df[MOCA_COL] > MIN_MOCA_SCORE]
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])

# Filter for Females

In [ ]:
df_high_vs_low = df_high_vs_low[df_high_vs_low['Gender'] == 'f']

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

## Blood Tests

In [ ]:
BLOOD_TESTS_SAVE_DIR = SAVE_DIR + 'blood_tests/'

### Hemoglobin_A1C

In [ ]:
HA1C = 'Hemoglobin_A1C'

df_high_vs_low_ha1c = df_high_vs_low.dropna(subset=[HA1C])


kdeplot_high_vs_low(df_high_vs_low_ha1c, HA1C, TAU_CAT, graph_colors, 'Hemoglobin A1C vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Hemoglobin_A1C_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Catagorical data
HA1C_INT = 5.7
HA1C_HIGH = 6.4

df_high_vs_low_ha1c['HA1C_CAT'] = pd.cut(df_high_vs_low_ha1c[HA1C], bins=[0, HA1C_INT, HA1C_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])


plot_histplot(df_high_vs_low_ha1c, 'HA1C_CAT', TAU_CAT, 3, graph_colors, 'Hemoglobin A1C vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Hemoglobin_A1C_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'Low'][HA1C]
group2 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'High'][HA1C]

p_ha1c = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'Low']['HA1C_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_ha1c[df_high_vs_low_ha1c[TAU_CAT] == 'High']['HA1C_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

p_ha1c_cat = stat_test(data_type, group1, group2)

### Cholesterol

In [ ]:
CHOL = 'Cholesterol'

df_high_vs_low_chol = df_high_vs_low.dropna(subset=[CHOL])


kdeplot_high_vs_low(df_high_vs_low_chol, CHOL, TAU_CAT, graph_colors, 'Cholesterol vs Tau 217 category', SAVE_DIR + 'Cholesterol_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Categorical data
CHOL_INT = 180
CHOL_HIGH = 240

df_high_vs_low_chol['CHOL_CAT'] = pd.cut(df_high_vs_low_chol[CHOL], bins=[0, CHOL_INT, CHOL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])


plot_histplot(df_high_vs_low_chol, 'CHOL_CAT', TAU_CAT, 3, graph_colors, 'Cholesterol vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Cholesterol_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'Low'][CHOL]
group2 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'High'][CHOL]

p_cholesterol = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'Low']['CHOL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_chol[df_high_vs_low_chol[TAU_CAT] == 'High']['CHOL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

p_cholesterol_cat = stat_test(data_type, group1, group2)

### Triglycerides

In [ ]:
TRIG = 'Triglycerides'

df_high_vs_low_trig = df_high_vs_low.dropna(subset=[TRIG])
df_high_vs_low_trig[TRIG][df_high_vs_low_trig[TRIG] == '38..8'] = 38.8


kdeplot_high_vs_low(df_high_vs_low_trig, TRIG, TAU_CAT, graph_colors, 'Triglycerides vs Tau 217 category',BLOOD_TESTS_SAVE_DIR + 'Triglycerides_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Catagorical Data
TRG_INT = 150
TRG_HIGH = 200

df_high_vs_low_trig['TRG_CAT'] = pd.cut(df_high_vs_low_trig[TRIG], bins=[0, TRG_INT, TRG_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])


plot_histplot(df_high_vs_low_trig, 'TRG_CAT', TAU_CAT, 3, graph_colors, 'Triglycerides vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Triglycerides_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'Low'][TRIG].astype(float)
group2 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'High'][TRIG].astype(float)

p_triglycerides = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'Low']['TRG_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_trig[df_high_vs_low_trig[TAU_CAT] == 'High']['TRG_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

p_triglycerides_cat = stat_test(data_type, group1, group2)

### LDL

In [ ]:
LDL = 'LDL'

df_high_vs_low_ldl = df_high_vs_low.dropna(subset=[LDL])


kdeplot_high_vs_low(df_high_vs_low_ldl, LDL, TAU_CAT, graph_colors, 'LDL vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'LDL_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
LDL_INT = 100
LDL_HIGH = 130

df_high_vs_low_ldl['LDL_CAT'] = pd.cut(df_high_vs_low_ldl[LDL], bins=[0, LDL_INT, LDL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])


plot_histplot(df_high_vs_low_ldl, 'LDL_CAT', TAU_CAT, 3, graph_colors, 'LDL vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'LDL_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'Low'][LDL]
group2 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'High'][LDL]

p_ldl = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'Low']['LDL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})
group2 = df_high_vs_low_ldl[df_high_vs_low_ldl[TAU_CAT] == 'High']['LDL_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2})

p_ldl_cat = stat_test(data_type, group1, group2)

### HDL

In [ ]:
HDL = 'HDL'

df_high_vs_low_hdl = df_high_vs_low.dropna(subset=[HDL])

kdeplot_high_vs_low(df_high_vs_low_hdl, HDL, TAU_CAT, graph_colors, 'HDL vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'HDL_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_hdl[df_high_vs_low_hdl[TAU_CAT] == 'Low'][HDL]
group2 = df_high_vs_low_hdl[df_high_vs_low_hdl[TAU_CAT] == 'High'][HDL]

p_hdl = stat_test(data_type, group1, group2)

### CRP

In [ ]:
df_high_vs_low['CRP'].replace({'<0.4': 0, '<0.5': 0, '<0.6':0}, inplace=True)

In [ ]:
# Cataogrical data
CRP = 'CRP'
CRP_INT = 0.5
CRP_HIGH = 1
CRP_VERY_HIGH = 2
# df_high_vs_low['CRP'] = df_high_vs_low['CRP'].astype(float)
df_high_vs_low_crp = df_high_vs_low.dropna(subset=[CRP])
df_high_vs_low_crp = df_high_vs_low_crp[df_high_vs_low_crp[CRP] != '<1']
df_high_vs_low_crp[CRP] = df_high_vs_low_crp[CRP].astype(float)
# df_high_vs_low_crp[CRP] = df_high_vs_low_crp[CRP].astype(float)
df_high_vs_low_crp['CRP_CAT'] = pd.cut(df_high_vs_low_crp[CRP], bins=[0, CRP_INT, CRP_HIGH, CRP_VERY_HIGH, np.inf], labels=['Low', 'Intermediate', 'High', 'Very High'])

plot_histplot(df_high_vs_low_crp, 'CRP_CAT', TAU_CAT, 4, graph_colors, 'CRP vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'CRP_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
df_high_vs_low_crp[CRP].unique()

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low'][CRP]
group2 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'High'][CRP]

p_crp = stat_test(data_type, group1, group2)

In [ ]:
df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).unique()
df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].unique()

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'Low']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).dropna()
group2 = df_high_vs_low_crp[df_high_vs_low_crp[TAU_CAT] == 'High']['CRP_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2, 'Very High': 3}).dropna()

p_crp_cat = stat_test(data_type, group1, group2)

### HB

In [ ]:
HB = 'HB'

df_high_vs_low_hb = df_high_vs_low.dropna(subset=[HB])

kdeplot_high_vs_low(df_high_vs_low_hb, HB, TAU_CAT, graph_colors, 'HB vs Tau 217 category',BLOOD_TESTS_SAVE_DIR + 'HB_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_hb[df_high_vs_low_hb[TAU_CAT] == 'Low'][HB]
group2 = df_high_vs_low_hb[df_high_vs_low_hb[TAU_CAT] == 'High'][HB]

p_hemoglobin = stat_test(data_type, group1, group2)

### RBC

In [ ]:
RBC = 'RBC'

df_high_vs_low_rbc = df_high_vs_low.dropna(subset=[RBC])

kdeplot_high_vs_low(df_high_vs_low_rbc, RBC, TAU_CAT, graph_colors, 'RBC vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'RBC_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_rbc[df_high_vs_low_rbc[TAU_CAT] == 'Low'][RBC]
group2 = df_high_vs_low_rbc[df_high_vs_low_rbc[TAU_CAT] == 'High'][RBC]

p_rbc = stat_test(data_type, group1, group2)

### WBC

In [ ]:
WBC = 'WBC'

df_high_vs_low_wbc = df_high_vs_low.dropna(subset=[WBC])

kdeplot_high_vs_low(df_high_vs_low_wbc, WBC, TAU_CAT, graph_colors, 'WBC vs Tau 217 category',BLOOD_TESTS_SAVE_DIR + 'WBC_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
WBC_LOW = 4.5
WBC_HIGH = 11

df_high_vs_low_wbc['WBC_CAT'] = pd.cut(df_high_vs_low_wbc[WBC], bins=[0, WBC_LOW, WBC_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])

plot_histplot(df_high_vs_low_wbc, 'WBC_CAT', TAU_CAT, 3, graph_colors, 'WBC vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'WBC_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_wbc[df_high_vs_low_wbc[TAU_CAT] == 'Low'][WBC]
group2 = df_high_vs_low_wbc[df_high_vs_low_wbc[TAU_CAT] == 'High'][WBC]

p_wbc = stat_test(data_type, group1, group2)

### Lymphocytes

In [ ]:
LYMPH = 'Lymphocytes'

df_high_vs_low_lymph = df_high_vs_low.dropna(subset=[LYMPH])

kdeplot_high_vs_low(df_high_vs_low_lymph, LYMPH, TAU_CAT, graph_colors, 'Lymphocytes vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Lymphocytes_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
LYMPH_LOW = 1.3
LYMPH_HIGH = 4.7

df_high_vs_low_lymph['LYMPH_CAT'] = pd.cut(df_high_vs_low_lymph[LYMPH], bins=[0, LYMPH_LOW, LYMPH_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])

plot_histplot(df_high_vs_low_lymph, 'LYMPH_CAT', TAU_CAT, 3, graph_colors, 'Lymphocytes vs Tau 217 category', BLOOD_TESTS_SAVE_DIR + 'Lymphocytes_categorize_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_lymph[df_high_vs_low_lymph[TAU_CAT] == 'Low'][LYMPH]
group2 = df_high_vs_low_lymph[df_high_vs_low_lymph[TAU_CAT] == 'High'][LYMPH]

stat_test(data_type, group1, group2)
p_lymphocytes = stat_test(data_type, group1, group2)

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_lymph[df_high_vs_low_lymph['LYMPH_CAT'] != 'High'] # Theres only two subjects with 'High' value
group1['LYMPH_CAT_num'] = group1['LYMPH_CAT'].replace({'Low': 0, 'Intermediate': 1, 'High': 2}).astype(int)
group2 = [TAU_CAT, 'LYMPH_CAT_num']

p_lymphocytes_cat = stat_test(data_type, group1, group2)

### Creatinine

In [ ]:
CREATININE = 'Creatinine'

df_high_vs_low_creatinine = df_high_vs_low.dropna(subset=[CREATININE])

kdeplot_high_vs_low(df_high_vs_low_creatinine, CREATININE, TAU_CAT, graph_colors, 'Creatinine vs Tau 217 category',BLOOD_TESTS_SAVE_DIR+ 'Creatinine_vs_Tau_217_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_creatinine[df_high_vs_low_creatinine[TAU_CAT] == 'Low'][CREATININE]
group2 = df_high_vs_low_creatinine[df_high_vs_low_creatinine[TAU_CAT] == 'High'][CREATININE]

p_creatinine = stat_test(data_type, group1, group2)

# Statistical Correction
I'm using BH-FDR

In [ ]:
ALPHA = 0.05
METHOD = 'fdr_bh'

In [ ]:
p_values = [p_ha1c, p_ha1c_cat, p_cholesterol, p_cholesterol_cat, p_triglycerides, p_triglycerides_cat, p_ldl, p_ldl_cat, p_hdl, p_crp, p_crp_cat, p_hemoglobin, p_rbc, p_wbc, p_lymphocytes, p_lymphocytes_cat, p_creatinine]
risk_factors = ['Hemoglobin A1C', 'Hemoglobin A1C (Categorical)', 'Cholesterol', 'Cholesterol (Categorical)', 'Triglycerides', 'Triglycerides (Categorical)', 'LDL', 'LDL (Categorical)', 'HDL', 'CRP', 'CRP (Categorical)', 'Hemoglobin', 'RBC', 'WBC', 'Lymphocytes', 'Lymphocytes (Categorical)', 'Creatinine']
p_values = [p for p in p_values if p is not None]
reject, p_values_corrected, _, _ = smm.multipletests(p_values, alpha=ALPHA, method=METHOD)

for i, p in enumerate(p_values):
    if p is not None:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

print("\nSignificant features:")
for i, p in enumerate(p_values):
    if p < ALPHA:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

In [ ]:
HIGH_TAU_217_THR

# Results
### 0.53
* No significant tests.

### 0.449
* No significant tests after correction.
* Lymphocytes were significant before correction p-value = 0.045 -> adjusted p-value = 0.477